# Voting Classifier from Scratch

## Objectives
- Understand ensemble voting.
- Implement hard voting and soft voting.
- Combine KNN, Decision Tree, and Logistic Regression.
- Evaluate and compare the models with Scikit-Learn.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
X, y = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=2,
    n_classes=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (800, 6)
Testing: (200, 6)


In [3]:
models = {
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=42
    ),
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )
}

for model in models.values():
    model.fit(X_train, y_train)

for name, model in models.items():
    print(f"{name}: {accuracy_score(y_test, model.predict(X_test)):.4f}")

KNN: 0.8850
Decision Tree: 0.8500
Logistic Regression: 0.7000


In [4]:
class VotingClassifierScratch:
    def __init__(self, models, voting="hard", weights=None):
        if voting not in ("hard", "soft"):
            raise ValueError("voting must be 'hard' or 'soft'.")

        if not models:
            raise ValueError("Provide at least one model.")

        self.models = models
        self.voting = voting
        self.weights = (
            np.ones(len(models), dtype=float)
            if weights is None else np.asarray(weights, dtype=float)
        )

        if (
            self.weights.shape != (len(models),)
            or not np.all(np.isfinite(self.weights))
            or np.any(self.weights < 0)
            or self.weights.sum() <= 0
        ):
            raise ValueError("Weights must be finite, non-negative, and not all zero.")

    def fit(self, X, y):
        self.classes_ = np.unique(y)

        if len(self.classes_) < 2:
            raise ValueError("Training data must contain at least two classes.")

        self.fitted_models_ = []

        for model in self.models:
            model.fit(X, y)
            self.fitted_models_.append(model)

        if self.voting == "soft":
            for model in self.fitted_models_:
                if not hasattr(model, "predict_proba"):
                    raise ValueError(
                        "Every model must support predict_proba for soft voting."
                    )
                if not np.array_equal(model.classes_, self.classes_):
                    raise ValueError("All models must use the same class labels.")

        return self

    def predict(self, X):
        if self.voting == "soft":
            probabilities = np.array([
                model.predict_proba(X)
                for model in self.fitted_models_
            ])

            avg_probabilities = np.average(
                probabilities, axis=0, weights=self.weights
            )

            return self.classes_[np.argmax(avg_probabilities, axis=1)]

        predictions = np.array([
            model.predict(X)
            for model in self.fitted_models_
        ])

        results = []

        for column in predictions.T:
            votes = np.array([
                np.sum(self.weights[column == cls])
                for cls in self.classes_
            ])
            results.append(self.classes_[np.argmax(votes)])

        return np.asarray(results)

In [5]:
hard_voter = VotingClassifierScratch(
    models=[
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        DecisionTreeClassifier(max_depth=5, random_state=42),
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    ],
    voting="hard"
)

hard_voter.fit(X_train, y_train)
hard_predictions = hard_voter.predict(X_test)

print("Hard Voting Accuracy:", accuracy_score(y_test, hard_predictions))

Hard Voting Accuracy: 0.875


In [6]:
soft_voter = VotingClassifierScratch(
    models=[
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        DecisionTreeClassifier(max_depth=5, random_state=42),
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    ],
    voting="soft"
)

soft_voter.fit(X_train, y_train)
soft_predictions = soft_voter.predict(X_test)

print("Soft Voting Accuracy:", accuracy_score(y_test, soft_predictions))

Soft Voting Accuracy: 0.87


In [7]:
results = []

for name, model in models.items():
    predictions = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0)
    })

for name, predictions in [
    ("Hard Voting (Scratch)", hard_predictions),
    ("Soft Voting (Scratch)", soft_predictions)
]:
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0)
    })

pd.DataFrame(results).set_index("Model").round(4)

,Accuracy,Precision,Recall,F1-score
Model,,,,
KNN,0.885,0.9053,0.86,0.8821
Decision Tree,0.850,0.8646,0.83,0.8469
Logistic Regression,0.700,0.7222,0.65,0.6842
Hard Voting (Scratch),0.875,0.9032,0.84,0.8705
Soft Voting (Scratch),0.870,0.9022,0.83,0.8646
